# IS 445 Homework: UFO Sightings with Altair

Two plots of the UFO sightings dataset, exported as vega-lite JSON for my Jekyll page.

1. How the shapes people report have changed over time (hover to highlight a shape).
2. When sightings happen by month and hour of day (dropdown to filter by shape).

In [1]:
import pandas as pd
import altair as alt
import os

## Read in the data

The file has no header row, so I'm passing in column names (same ones we used in class).

In [2]:
ufo_url = 'https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/ufo-scrubbed-geocoded-time-standardized-00.csv'

ufos = pd.read_csv(ufo_url,
                   names=['date', 'city', 'state', 'country',
                          'shape', 'duration_seconds', 'duration',
                          'comment', 'report_date',
                          'latitude', 'longitude'],
                   low_memory=False)
ufos.head()

,date,city,state,country,shape,duration_seconds,duration,comment,report_date,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700.0,45 minutes,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.384210,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.200000,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833
4,10/10/1960 20:00,kaneohe,hi,us,light,900.0,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611


In [3]:
len(ufos)

80332

## Clean up

- Turn the `date` strings into real timestamps so I can pull out the year, month, and hour.
- Drop rows with no shape listed, since both plots are split by shape.
- Keep 1950 on. There are only a few dozen reports before that.

In [4]:
ufos['date'] = pd.to_datetime(ufos['date'], format='%m/%d/%Y %H:%M')

ufos['year'] = ufos['date'].dt.year
ufos['month'] = ufos['date'].dt.month
ufos['hour'] = ufos['date'].dt.hour

ufos_clean = ufos.dropna(subset=['shape'])
ufos_clean = ufos_clean[ufos_clean['year'] >= 1950].copy()
len(ufos_clean)

78283

There are 29 different shapes, and most of them are tiny. I'm keeping six shapes that show a clear trend over time and putting everything else (including the dataset's own "other" and "unknown" labels) into one "all other shapes" group.

In [5]:
ufos_clean['shape'].value_counts()

shape
light        16556
triangle      7863
circle        7599
fireball      6205
other         5642
unknown       5576
sphere        5379
disk          5178
oval          3724
formation     2452
cigar         2046
changing      1962
flash         1327
rectangle     1293
cylinder      1278
diamond       1178
chevron        951
egg            759
teardrop       750
cone           316
cross          233
delta            7
round            2
crescent         2
pyramid          1
flare            1
hexagon          1
dome             1
changed          1
Name: count, dtype: int64

In [6]:
keep_shapes = ['light', 'fireball', 'circle', 'triangle', 'disk', 'cigar']

ufos_clean['shape_group'] = ufos_clean['shape'].where(ufos_clean['shape'].isin(keep_shapes),
                                                      'all other shapes')
shape_order = keep_shapes + ['all other shapes']
ufos_clean['shape_group'].value_counts()

shape_group
all other shapes    32836
light               16556
triangle             7863
circle               7599
fireball             6205
disk                 5178
cigar                2046
Name: count, dtype: int64

## Plot 1: Share of reports by shape, in 5-year periods

Single years before the mid-90s only have a couple hundred reports, so the yearly shares jump around a lot. Grouping into 5-year periods smooths that out. For each period I count reports per shape group and divide by the period total to get a percent.

In [7]:
ufos_clean['period'] = (ufos_clean['year'] // 5) * 5

shape_share = ufos_clean.groupby(['period', 'shape_group']).size().reset_index(name='reports')
shape_share['percent'] = 100 * shape_share['reports'] / shape_share.groupby('period')['reports'].transform('sum')

# labels for the tooltip, and a real date so altair treats x as temporal
shape_share['period_label'] = shape_share['period'].astype(str) + '-' + (shape_share['period'] + 4).astype(str)
shape_share['period_start'] = pd.to_datetime(shape_share['period'].astype(str), format='%Y')

# stacking order: light on the bottom, "all other shapes" on top
shape_share['stack_order'] = shape_share['shape_group'].map({s: i for i, s in enumerate(shape_order)})
shape_share.head(10)

,period,shape_group,reports,percent,period_label,period_start,stack_order
0,1950,all other shapes,56,31.638418,1950-1954,1950-01-01,6
1,1950,cigar,12,6.779661,1950-1954,1950-01-01,5
2,1950,circle,23,12.994350,1950-1954,1950-01-01,2
3,1950,disk,64,36.158192,1950-1954,1950-01-01,4
4,1950,fireball,7,3.954802,1950-1954,1950-01-01,1
5,1950,light,11,6.214689,1950-1954,1950-01-01,0
6,1950,triangle,4,2.259887,1950-1954,1950-01-01,3
7,1955,all other shapes,81,34.033613,1955-1959,1955-01-01,6
8,1955,cigar,21,8.823529,1955-1959,1955-01-01,5
9,1955,circle,24,10.084034,1955-1959,1955-01-01,2


In [8]:
# the last period is 2010-2014, but the data stops in May 2014
ufos_clean['date'].max()

Timestamp('2014-05-08 18:45:00')

In [9]:
hover = alt.selection_point(fields=['shape_group'], on='mouseover', clear='mouseout')

shapes_over_time = alt.Chart(shape_share).mark_area().encode(
    x=alt.X('period_start:T', axis=alt.Axis(format='%Y', title='Start of 5-year period')),
    y=alt.Y('percent:Q', stack='zero', title='Share of reports (%)', scale=alt.Scale(domain=[0, 100])),
    color=alt.Color('shape_group:N', sort=shape_order,
                    scale=alt.Scale(scheme='tableau10'),
                    legend=alt.Legend(title='Reported shape')),
    order=alt.Order('stack_order:Q'),
    opacity=alt.condition(hover, alt.value(1.0), alt.value(0.25)),
    tooltip=[alt.Tooltip('shape_group:N', title='Shape'),
             alt.Tooltip('period_label:N', title='Years'),
             alt.Tooltip('percent:Q', title='Share (%)', format='.1f'),
             alt.Tooltip('reports:Q', title='Reports')]
).add_params(
    hover
).properties(
    width=600,
    height=350,
    title='What shape did people say the UFO was? (1950-2014)'
)

shapes_over_time

alt.Chart(...)

## Plot 2: When sightings happen (month x hour of day)

I count reports for every combination of shape group, month, and hour. That keeps the shape group in the data so the dropdown can filter on it, but the table is only about 2,000 rows instead of ~78,000. Altair then sums the counts for whatever is left after the filter.

In [10]:
month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

time_counts = ufos_clean.groupby(['shape_group', 'month', 'hour']).size().reset_index(name='reports')
time_counts['month_name'] = time_counts['month'].apply(lambda m: month_names[m - 1])
time_counts.head()

,shape_group,month,hour,reports,month_name
0,all other shapes,1,0,214,Jan
1,all other shapes,1,1,93,Jan
2,all other shapes,1,2,66,Jan
3,all other shapes,1,3,59,Jan
4,all other shapes,1,4,53,Jan


In [11]:
len(time_counts)

2002

In [12]:
# dropdown: None = no selection = show everything
shape_dropdown = alt.binding_select(options=[None] + shape_order,
                                    labels=['all shapes'] + shape_order,
                                    name='Shape: ')
shape_select = alt.selection_point(fields=['shape_group'], bind=shape_dropdown)

time_heatmap = alt.Chart(time_counts).mark_rect().encode(
    x=alt.X('hour:O', title='Hour of day (0 = midnight)', axis=alt.Axis(labelAngle=0)),
    y=alt.Y('month_name:O', sort=month_names, title='Month'),
    color=alt.Color('sum(reports):Q', scale=alt.Scale(scheme='yelloworangered'),
                    legend=alt.Legend(title='Reports')),
    tooltip=[alt.Tooltip('month_name:O', title='Month'),
             alt.Tooltip('hour:O', title='Hour'),
             alt.Tooltip('sum(reports):Q', title='Reports')]
).transform_filter(
    shape_select
).add_params(
    shape_select
).properties(
    width=600,
    height=300,
    title='UFO reports by month and hour of day'
)

time_heatmap

alt.Chart(...)

A couple of quick checks on what the heatmap shows. First, the busiest hour for each month (it moves later in the summer). Second, the fireball reports by day of the year, since July looks extra hot when you filter to fireball.

In [13]:
time_counts.groupby(['month', 'hour'])['reports'].sum().unstack().idxmax(axis=1)

month
1     20
2     20
3     20
4     21
5     21
6     22
7     22
8     22
9     21
10    20
11    19
12    19
dtype: int32

In [14]:
fireballs = ufos_clean[ufos_clean['shape'] == 'fireball']
fireballs['date'].dt.strftime('%m-%d').value_counts().head()

date
07-04    193
11-16     94
01-01     70
12-31     54
08-15     51
Name: count, dtype: int64

## Save the specs for Jekyll

After these run, move the JSON files into `assets/json` in the Jekyll repo.

In [15]:
myJekyllDir = './'

shapes_over_time.save(myJekyllDir + 'ufo_shapes_over_time.json')
time_heatmap.save(myJekyllDir + 'ufo_time_heatmap.json')

In [16]:
# file sizes in bytes -- both small because I aggregated before plotting
os.stat(myJekyllDir + 'ufo_shapes_over_time.json').st_size, os.stat(myJekyllDir + 'ufo_time_heatmap.json').st_size

(17478, 177547)